## ML_1025: SwiGLU MLP

**Difficulty**: Medium | **TorchCode**: #15

### Problem
Implement the SwiGLU MLP used in LLaMA/PaLM. Three linear projections: gate, up, and down. The gate is passed through SiLU and element-wise multiplied with the up projection before the down projection.

### Formula
$$\text{SwiGLU}(x) = W_{\text{down}}\,\bigl(\text{SiLU}(W_{\text{gate}}\, x) \odot W_{\text{up}}\, x\bigr)$$
$$\text{SiLU}(x) = x \cdot \sigma(x)$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj   = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Test 1: Parameter shapes ---
mlp = SwiGLUMLP(d_model=64, d_ff=128)
assert isinstance(mlp, nn.Module)
assert mlp.gate_proj.weight.shape == (128, 64)
assert mlp.up_proj.weight.shape   == (128, 64)
assert mlp.down_proj.weight.shape == (64, 128)

# --- Test 2: Forward output shape ---
mlp2 = SwiGLUMLP(d_model=32, d_ff=64)
assert mlp2(torch.randn(2, 8, 32)).shape == (2, 8, 32)

# --- Test 3: Numerical correctness ---
torch.manual_seed(0)
mlp3 = SwiGLUMLP(d_model=16, d_ff=32)
x = torch.randn(1, 4, 16)
out = mlp3(x)
ref = mlp3.down_proj(F.silu(mlp3.gate_proj(x)) * mlp3.up_proj(x))
assert torch.allclose(out, ref, atol=1e-5)

# --- Test 4: 2-D input ---
assert SwiGLUMLP(32, 64)(torch.randn(4, 32)).shape == (4, 32)

# --- Test 5: Gradient flow ---
x = torch.randn(2, 4, 32, requires_grad=True)
SwiGLUMLP(32, 64)(x).sum().backward()
assert x.grad is not None

print('All tests passed!')